# Transfer Learning ve Fine-Tuning ile İleri Seviye Görüntü Sınıflandırma

Bu modül; sıfırdan büyük bir sinir ağı eğitmek yerine, milyonlarca görsel içeren **ImageNet** veri kümesi üzerinde önceden eğitilmiş (pre-trained) modern mimarilerin (**MobileNetV2 / ResNet50**) ağırlıklarını kullanarak transfer öğrenme ve ince ayar (fine-tuning) uygulama metodolojisini açıklar.

---

## 1. Transfer Learning Mimarisi ve Matematiksel Mantık

Derin evrişimli sinir ağlarında katmanlar hiyerarşik öznitelikler öğrenir:
- **İlk Katmanlar:** Genel öznitelikler (kenarlar, renk geçişleri, basit dokular). Tüm görsel veri kümeleri için evrenseldir.
- **Orta Katmanlar:** Şekil parçaları, karmaşık desenler ve dokular.
- **Son Katmanlar:** Veri kümesine özgü üst düzey anlamsal nesneler (örneğin kedi kulağı, araba tekerleği).

**Strateji:**
1. Taban modelin (Base Model) tüm katmanları dondurulur (`trainable = False`): Ağırlık gradyanları güncellenmez.
2. Taban modelin üzerine özel bir sınıflandırma başlığı eklenir:
   $$y = \text{Softmax}(W_2 \cdot \text{Dropout}(\text{ReLU}(W_1 \cdot \text{GAP}(F) + b_1)) + b_2)$$
   Burada $\text{GAP}$ (Global Average Pooling), $H \times W \times C$ tensörünü $1 \times C$ vektörüne indirger.
3. Özel başlık eğitildikten sonra, taban modelin son birkaç katmanı çözülür (`trainable = True`) ve çok düşük bir öğrenme oranıyla (örneğin $\alpha = 10^{-5}$) **Fine-Tuning** yapılır.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow Versiyonu: {tf.__version__}")


## 2. Önceden Eğitilmiş MobileNetV2 Taban Modelinin Kurulumu

In [ ]:
# Giriş çözünürlüğü ve sınıf sayısı
IMG_SHAPE = (160, 160, 3)
NUM_CLASSES = 5

# ImageNet ağırlıkları ile MobileNetV2 yükleme (Sınıflandırıcı başlığı hariç)
base_model = keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights='imagenet'
)

# Aşama 1: Taban modelin katmanlarını dondurma
base_model.trainable = False

print(f"Taban model toplam katman sayısı: {len(base_model.layers)}")
print(f"Eğitilebilir parametre sayısı: {sum([tf.size(w).numpy() for w in base_model.trainable_weights])}")


## 3. Özel Sınıflandırma Başlığının İnşası

In [ ]:
inputs = keras.Input(shape=IMG_SHAPE)

# Veri artırma (Data Augmentation) katmanları
x = layers.RandomFlip('horizontal')(inputs)
x = layers.RandomRotation(0.1)(x)

# MobileNetV2 ön işleme (Pikselleri [-1, 1] aralığına ölçekleme)
x = keras.applications.mobilenet_v2.preprocess_input(x)

# Dondurulmuş taban modelden geçirme (training=False batch norm istatistiklerini korur)
x = base_model(x, training=False)

# Global Average Pooling ve Sınıflandırıcı
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


## 4. Aşama 2: Fine-Tuning (İnce Ayar) Stratejisi

Özel başlık eğitildikten sonra, taban modelin en üstteki 20-30 katmanını çözerek hedef veri kümesine özgü karmaşık desenleri ince ayarlarız.


In [ ]:
# Taban modeli eğitilebilir yapma
base_model.trainable = True

# İlk 100 katmanı dondurup, son katmanları çözme
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"İnce ayar yapılacak eğitilebilir katman sayısı: {len(base_model.trainable_weights)}")

# DİKKAT: İnce ayarda öğrenme oranı 10 kat ila 100 kat daha düşük seçilmelidir
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model fine-tuning için hazırlandı!")


## 5. Mühendislik Çıkarımları ve En İyi Uygulamalar

1. **Batch Normalization Koruması:** `base_model(x, training=False)` çağrısı, transfer learning sırasında ImageNet'in hareketli ortalama ve varyans istatistiklerinin bozulmasını önler.
2. **Öğrenme Oranı (Learning Rate):** Fine-tuning sırasında yüksek öğrenme oranı kullanılırsa taban modelin öğrendiği genel öznitelikler silinir (Catastrophic Forgetting). Bu yüzden $\le 10^{-5}$ aralığı tercih edilir.
3. **Veri Kısıtı:** Elinizde 100-500 adet gibi az sayıda görsel varsa sıfırdan model eğitmek yerine Transfer Learning zorunlu bir endüstri standardıdır.
